# Project 01 — Missile Geometry 101
## World Defense Organization (WDO)

**Analyst:** Aakash Harolia 
**Base Location:** Dallas, TX (32.7767, -96.7970)

**Mission:** Analyze incoming non-human threats using spatial geometry.  
We do not fire weapons. We trust geometry.

## Setup — Imports, Paths, Constants

In [ ]:
from pathlib import Path
import json
import csv
import folium
from shapely.geometry import shape, LineString, Point

ROOT = Path.cwd() if (Path.cwd() / 'src' / 'wdo').exists() else (Path.cwd() / 'Assignments_Completed' / 'Project_01')
assert (ROOT / 'src').exists(), f'src/ not found. ROOT={ROOT}'
assert (ROOT / 'data').exists(), f'data/ not found. ROOT={ROOT}'

from src.wdo.io_shapefile import shapefile_to_features
from src.wdo.viz_map      import make_base_map, add_geojson_layer, add_base_marker
from src.wdo.geo_math     import haversine_km, initial_bearing_deg, destination_point, trajectory_points

MAPS_DIR = ROOT / 'maps'
MAPS_DIR.mkdir(exist_ok=True)

BASE_LAT = 32.7767
BASE_LON = -96.7970
BASE_LABEL = 'WDO Base — Dallas'

TYPE_COLOR = {'alien': 'green', 'orbital': 'purple', 'airborne': 'orange', 'kaiju': 'red'}
DAMAGE_RADIUS_KM = {'kaiju': 100, 'airborne': 200, 'alien': 300, 'orbital': 600}
SEVERITY = {'kaiju': 'high', 'airborne': 'medium', 'alien': 'high', 'orbital': 'critical'}
DANGER_RADIUS_KM = 500.0

def add_legend(m, html):
    m.get_root().html.add_child(folium.Element(html))

print('OK — toolkit loaded')

## Load shared data + compute trajectories

In [ ]:
SHP_PATH = ROOT / 'data' / 'world_borders' / 'world_borders.shp'
features = shapefile_to_features(SHP_PATH, id_field=None)

candidate_fields = ['NAME', 'ADMIN', 'CNTRY_NAME', 'NAME_LONG', 'name']
available = set(features[0]['properties'].keys())
tooltip_field = next((f for f in candidate_fields if f in available), None)
name_field = tooltip_field or list(features[0]['properties'].keys())[0]

with open(ROOT / 'data' / 'threats' / 'threats.json') as f:
    threats = json.load(f)

for t in threats:
    t['distance_km'] = haversine_km(t['origin_lat'], t['origin_lon'], BASE_LAT, BASE_LON)
    t['bearing_to_base_deg'] = initial_bearing_deg(t['origin_lat'], t['origin_lon'], BASE_LAT, BASE_LON)

closest = min(threats, key=lambda t: t['distance_km'])

STEP_MIN = 2.0
for t in threats:
    pts = trajectory_points(
        origin_lat=t['origin_lat'], origin_lon=t['origin_lon'],
        bearing_deg=t['bearing_deg'], speed_kmh=t['speed_kmh'],
        duration_min=t['duration_min'], step_min=STEP_MIN,
    )
    t['trajectory'] = pts
    t['endpoint']   = pts[-1]
    t['total_km']   = round(t['speed_kmh'] * (t['duration_min'] / 60.0), 1)

country_geoms = []
for idx, f in enumerate(features):
    try:
        geom = shape(f['geometry'])
        if not geom.is_valid:
            geom = geom.buffer(0)
    except Exception:
        continue
    cname = f['properties'].get(name_field, f'feature_{idx}')
    country_geoms.append((cname, geom, idx))

print(f'Loaded {len(features)} countries, {len(threats)} threats, {len(country_geoms)} geometries')

---
## Milestone 1 — Plot the World

In [ ]:
m1 = make_base_map(BASE_LAT, BASE_LON, zoom=3, tiles='OpenStreetMap')
add_geojson_layer(m1, features, name='World Borders', tooltip_field=tooltip_field)
add_base_marker(m1, BASE_LAT, BASE_LON, label=BASE_LABEL)

M1_LEGEND = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px; border: 2px solid grey;
            font-family: sans-serif; font-size: 13px;">
  <b>Milestone 1 — Base Map</b><br>
  &#127968; WDO Base (Dallas)<br>
  &#9633; world country borders
</div>
'''
add_legend(m1, M1_LEGEND)
folium.LayerControl().add_to(m1)
m1.save(str(MAPS_DIR / 'milestone_01_base_map.html'))
print('Saved Milestone 1 map')
m1

---
## Milestone 2 — Distance & Bearing

In [ ]:
print(f"{'ID':<6} {'Type':<10} {'Dist (km)':>10} {'Bearing':>10}")
print('-' * 40)
for t in sorted(threats, key=lambda x: x['distance_km']):
    print(f"{t['id']:<6} {t['type']:<10} {t['distance_km']:>10.0f} {t['bearing_to_base_deg']:>9.1f}")

m2 = make_base_map(BASE_LAT, BASE_LON, zoom=3, tiles='OpenStreetMap')
add_geojson_layer(m2, features, name='World Borders', tooltip_field=tooltip_field)
add_base_marker(m2, BASE_LAT, BASE_LON, label=BASE_LABEL)
for t in threats:
    color = TYPE_COLOR.get(t['type'], 'gray')
    is_closest = (t['id'] == closest['id'])
    folium.CircleMarker(
        location=[t['origin_lat'], t['origin_lon']],
        radius=9 if is_closest else 6,
        color='black' if is_closest else color,
        weight=3 if is_closest else 1,
        fill=True, fill_color=color, fill_opacity=0.85,
        tooltip=f"{t['id']} ({t['type']}) — {t['distance_km']:.0f} km",
    ).add_to(m2)

M2_LEGEND = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px; border: 2px solid grey;
            font-family: sans-serif; font-size: 13px;">
  <b>Threat type</b><br>
  <span style="color:green;">&#9679;</span> alien<br>
  <span style="color:purple;">&#9679;</span> orbital<br>
  <span style="color:orange;">&#9679;</span> airborne<br>
  <span style="color:red;">&#9679;</span> kaiju<br>
  <span style="color:black;">&#9711;</span> closest threat
</div>
'''
add_legend(m2, M2_LEGEND)
folium.LayerControl().add_to(m2)
m2.save(str(MAPS_DIR / 'milestone_02_distance_bearing.html'))
print('Saved Milestone 2 map')
m2

---
## Milestone 3 — Trajectories (Point → Line)

In [ ]:
m3 = make_base_map(BASE_LAT, BASE_LON, zoom=3, tiles='OpenStreetMap')
add_geojson_layer(m3, features, name='World Borders', tooltip_field=tooltip_field)
add_base_marker(m3, BASE_LAT, BASE_LON, label=BASE_LABEL)
for t in threats:
    color = TYPE_COLOR.get(t['type'], 'gray')
    folium.PolyLine(
        locations=t['trajectory'], color=color, weight=3, opacity=0.8,
        tooltip=f"{t['id']} ({t['type']}) — {t['total_km']} km",
    ).add_to(m3)
    folium.CircleMarker(location=[t['origin_lat'], t['origin_lon']], radius=5,
        color=color, fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=f"{t['id']} origin").add_to(m3)
    folium.CircleMarker(location=t['endpoint'], radius=6, color=color, weight=2,
        fill=True, fill_color='white', fill_opacity=0.9,
        tooltip=f"{t['id']} endpoint").add_to(m3)

M3_LEGEND = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px; border: 2px solid grey;
            font-family: sans-serif; font-size: 13px;">
  <b>Threat type</b><br>
  <span style="color:green;">&#9679;</span> alien<br>
  <span style="color:purple;">&#9679;</span> orbital<br>
  <span style="color:orange;">&#9679;</span> airborne<br>
  <span style="color:red;">&#9679;</span> kaiju<br>
  <hr style="margin:4px 0;">
  &#9679; filled = origin<br>
  &#9711; open = endpoint
</div>
'''
add_legend(m3, M3_LEGEND)
folium.LayerControl().add_to(m3)
m3.save(str(MAPS_DIR / 'milestone_03_trajectories.html'))
print('Saved Milestone 3 map')
m3

---
## Milestone 4 — Intersections & Borders

In [ ]:
intersected_country_indices = set()
print(f"{'ID':<6} {'Type':<10} {'Min dist (km)':>14} {'Danger':>8}  Crosses")
print('-' * 80)
for t in threats:
    line = LineString([(lon, lat) for (lat, lon) in t['trajectory']])
    hits = []
    for cname, cgeom, fidx in country_geoms:
        if line.intersects(cgeom):
            hits.append(cname)
            intersected_country_indices.add(fidx)
    t['intersects_countries'] = hits
    min_dist = min(haversine_km(lat, lon, BASE_LAT, BASE_LON) for (lat, lon) in t['trajectory'])
    t['min_distance_to_base_km'] = min_dist
    t['danger'] = min_dist <= DANGER_RADIUS_KM
    short_hits = ', '.join(hits[:3]) + ('...' if len(hits) > 3 else '') if hits else '(none)'
    print(f"{t['id']:<6} {t['type']:<10} {min_dist:>14.0f} {'YES' if t['danger'] else 'no':>8}  {short_hits}")

m4 = make_base_map(BASE_LAT, BASE_LON, zoom=3, tiles='OpenStreetMap')
add_geojson_layer(m4, features, name='World Borders', tooltip_field=tooltip_field)
intersected_features = [features[i] for i in sorted(intersected_country_indices)]
if intersected_features:
    folium.GeoJson(
        {'type': 'FeatureCollection', 'features': intersected_features},
        name='Intersected Countries',
        style_function=lambda x: {'fillColor': '#ff4d4d', 'color': '#cc0000', 'weight': 2, 'fillOpacity': 0.4},
        tooltip=folium.GeoJsonTooltip(fields=[name_field]) if name_field else None,
    ).add_to(m4)
folium.Circle(location=[BASE_LAT, BASE_LON], radius=DANGER_RADIUS_KM * 1000,
    color='red', weight=2, fill=True, fill_color='red', fill_opacity=0.08,
    tooltip=f'Danger zone — {DANGER_RADIUS_KM:.0f} km').add_to(m4)
for t in threats:
    color = TYPE_COLOR.get(t['type'], 'gray')
    folium.PolyLine(locations=t['trajectory'], color=color,
        weight=5 if t['danger'] else 2,
        opacity=0.95 if t['danger'] else 0.6,
        tooltip=f"{t['id']} - {'DANGER' if t['danger'] else 'OK'}",
    ).add_to(m4)
    folium.CircleMarker(location=[t['origin_lat'], t['origin_lon']], radius=4,
        color=color, fill=True, fill_color=color, fill_opacity=0.9).add_to(m4)
add_base_marker(m4, BASE_LAT, BASE_LON, label=BASE_LABEL)

M4_LEGEND = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px; border: 2px solid grey;
            font-family: sans-serif; font-size: 13px;">
  <b>Milestone 4 - Intersections</b><br>
  <span style="background:#ff4d4d; color:#ff4d4d;">&nbsp;&nbsp;</span> countries crossed<br>
  <span style="color:red;">&#9711;</span> 500 km danger zone<br>
  <hr style="margin:4px 0;">
  thick line = enters danger zone<br>
  thin line = passes outside
</div>
'''
add_legend(m4, M4_LEGEND)
folium.LayerControl().add_to(m4)
m4.save(str(MAPS_DIR / 'milestone_04_intersections.html'))
print('Saved Milestone 4 map')
m4

---
## Milestone 5 — Damage Zones

In [ ]:
KM_PER_DEG = 111.0
damage_rows = []
for t in threats:
    radius_km = DAMAGE_RADIUS_KM.get(t['type'], 200)
    radius_deg = radius_km / KM_PER_DEG
    end_lat, end_lon = t['endpoint']
    blast_zone = Point(end_lon, end_lat).buffer(radius_deg)
    affected = []
    for cname, cgeom, fidx in country_geoms:
        if blast_zone.intersects(cgeom):
            affected.append(cname)
            damage_rows.append({
                'country': cname, 'threat_id': t['id'], 'threat_type': t['type'],
                'severity': SEVERITY.get(t['type'], 'unknown'), 'radius_km': radius_km,
            })
    t['blast_radius_km']    = radius_km
    t['affected_countries'] = affected

csv_path = ROOT / 'damage_zones.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['country', 'threat_id', 'threat_type', 'severity', 'radius_km'])
    writer.writeheader()
    writer.writerows(damage_rows)

print(f"{'ID':<6} {'Type':<10} {'Radius':>8} {'Severity':>10}  Countries hit")
print('-' * 80)
for t in threats:
    countries = ', '.join(t['affected_countries']) if t['affected_countries'] else '(open ocean)'
    print(f"{t['id']:<6} {t['type']:<10} {t['blast_radius_km']:>7}km {SEVERITY.get(t['type'], '-'):>10}  {countries}")
print(f'\nWrote {len(damage_rows)} rows to damage_zones.csv')

m5 = make_base_map(BASE_LAT, BASE_LON, zoom=3, tiles='OpenStreetMap')
add_geojson_layer(m5, features, name='World Borders', tooltip_field=tooltip_field)
affected_country_names = {row['country'] for row in damage_rows}
if affected_country_names:
    affected_features = [f for f in features if f['properties'].get(name_field) in affected_country_names]
    folium.GeoJson(
        {'type': 'FeatureCollection', 'features': affected_features},
        name='Countries in damage zones',
        style_function=lambda x: {'fillColor': '#ff9900', 'color': '#cc6600', 'weight': 2, 'fillOpacity': 0.4},
        tooltip=folium.GeoJsonTooltip(fields=[name_field]) if name_field else None,
    ).add_to(m5)
for t in threats:
    color = TYPE_COLOR.get(t['type'], 'gray')
    folium.PolyLine(locations=t['trajectory'], color=color, weight=2, opacity=0.7).add_to(m5)
    folium.CircleMarker(location=[t['origin_lat'], t['origin_lon']], radius=4,
        color=color, fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=f"{t['id']} origin").add_to(m5)
    end_lat, end_lon = t['endpoint']
    folium.Circle(
        location=[end_lat, end_lon], radius=t['blast_radius_km'] * 1000,
        color=color, weight=1, fill=True, fill_color=color, fill_opacity=0.25,
        tooltip=f"{t['id']} ({t['type']}) - {t['blast_radius_km']} km",
    ).add_to(m5)
add_base_marker(m5, BASE_LAT, BASE_LON, label=BASE_LABEL)

M5_LEGEND = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px; border: 2px solid grey;
            font-family: sans-serif; font-size: 13px; line-height: 1.5;">
  <b>Milestone 5 - Damage Zones</b><br>
  <span style="color:green;">&#11044;</span> alien - 300 km / high<br>
  <span style="color:purple;">&#11044;</span> orbital - 600 km / critical<br>
  <span style="color:orange;">&#11044;</span> airborne - 200 km / medium<br>
  <span style="color:red;">&#11044;</span> kaiju - 100 km / high<br>
  <hr style="margin:4px 0;">
  <span style="background:#ff9900; color:#ff9900;">&nbsp;&nbsp;</span> countries in damage zones
</div>
'''
add_legend(m5, M5_LEGEND)
folium.LayerControl().add_to(m5)
m5.save(str(MAPS_DIR / 'milestone_05_damage_zones.html'))
print('Saved Milestone 5 map')
m5